# Multi-Provider LLM Client — Demo

This notebook demonstrates the `llm_client` module:
1. Same prompt sent to all 3 providers with a comparative table.
2. Structured output with a Pydantic schema.
3. Streaming with one provider.
4. Cost tracking: multiple calls logged to `costs.jsonl` with analysis.

> **Prerequisites**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, and `GEMINI_API_KEY` in `../../.env` (repo root).

In [3]:
import sys
from pathlib import Path

# Ensure the package is importable when running from 04_multi_provider/
sys.path.insert(0, str(Path.cwd()))

from llm_client import (
    AnthropicProvider,
    GeminiProvider,
    LLMResponse,
    OpenAIProvider,
    StreamingResponse,
)

## 1 — Same prompt, all 3 providers

In [4]:
PROMPT = "Explain what a neural network is in 2 sentences."

openai = OpenAIProvider()
anthropic = AnthropicProvider()
gemini = GeminiProvider()

results: list[LLMResponse] = []

r_openai = await openai.generate(prompt=PROMPT, model="gpt-5.4")
r_anthropic = await anthropic.generate(prompt=PROMPT, model="claude-sonnet-4-6")
r_gemini = await gemini.generate(prompt=PROMPT, model="gemini-3.1-pro")

results = [r_openai, r_anthropic, r_gemini]
print("Calls complete.")

Calls complete.


In [5]:
import pandas as pd

rows = [
    {
        "provider": r.provider,
        "model": r.model,
        "response_text": r.text[:120] + "..." if len(r.text) > 120 else r.text,
        "input_tokens": r.input_tokens,
        "output_tokens": r.output_tokens,
        "cost_usd": f"${r.cost_usd:.6f}",
        "latency_ms": r.latency_ms,
    }
    for r in results
]

df = pd.DataFrame(rows)
df

,provider,model,response_text,input_tokens,output_tokens,cost_usd,latency_ms
0,openai,gpt-5.4,A neural network is a type of machine learning...,17,60,$0.000942,1983
1,anthropic,claude-sonnet-4-6,A neural network is a computational model insp...,20,60,$0.000960,1630
2,gemini,gemini-3.1-pro-preview,A neural network is a type of artificial intel...,12,53,$0.000660,6684


## 2 — Structured output with a Pydantic schema

In [6]:
from pydantic import BaseModel


class NeuralNetworkSummary(BaseModel):
    """Structured summary of the neural network explanation."""

    definition: str
    analogy: str
    use_case: str


STRUCTURED_PROMPT = (
    "Explain what a neural network is. "
    "Provide: a one-sentence definition, an analogy, and a real-world use case."
)

structured_result = await anthropic.generate(
    prompt=STRUCTURED_PROMPT,
    model="claude-sonnet-4-6",
    schema=NeuralNetworkSummary,
)

print("Raw text:", structured_result.text[:200])
print()
print("Parsed object:", structured_result.parsed)
print()
if structured_result.parsed:
    s: NeuralNetworkSummary = structured_result.parsed
    print(f"Definition : {s.definition}")
    print(f"Analogy    : {s.analogy}")
    print(f"Use case   : {s.use_case}")

Raw text: {"definition":"A neural network is a computational model inspired by the human brain, consisting of layers of interconnected nodes (neurons) that process and learn patterns from data to make predictio

Parsed object: definition='A neural network is a computational model inspired by the human brain, consisting of layers of interconnected nodes (neurons) that process and learn patterns from data to make predictions or decisions.' analogy='A neural network is like a team of specialists in an assembly line: each worker (neuron) receives input, processes it in their own way, and passes their result to the next worker, until the final worker delivers a finished product — with the whole team improving over time through feedback and practice.' use_case='A real-world use case is image recognition in healthcare, where neural networks analyze medical scans (such as X-rays or MRIs) to detect diseases like cancer with high accuracy, assisting doctors in making faster and more reliable dia

## 3 — Streaming with OpenAI

In [7]:
import sys

stream_result = await openai.generate(
    prompt=PROMPT,
    model="gpt-5.4",
    stream=True,
)

assert isinstance(stream_result, StreamingResponse)

print("Streaming response (token by token):")
print("-" * 50)

async for chunk in stream_result:
    print(chunk, end="", flush=True)

print()
print("-" * 50)

final = stream_result.final_response
print(f"\nFinal response assembled: {len(final.text)} chars")

Streaming response (token by token):
--------------------------------------------------
A neural network is a computer system inspired by the brain that learns patterns by adjusting connections between many simple processing units called neurons. It takes input data, processes it through layers, and improves at tasks like recognizing images or understanding text by learning from examples.
--------------------------------------------------

Final response assembled: 303 chars


## 4 — Cost tracking across providers

In [9]:
from llm_client import CostTracker

tracker = CostTracker(log_path=Path("costs.jsonl"))

openai_tracked = OpenAIProvider(cost_tracker=tracker)
anthropic_tracked = AnthropicProvider(cost_tracker=tracker)
gemini_tracked = GeminiProvider(cost_tracker=tracker)

COST_PROMPT = "What is backpropagation? Answer in one sentence."

await openai_tracked.generate(prompt=COST_PROMPT, model="gpt-5.4-nano")
await openai_tracked.generate(prompt=COST_PROMPT, model="gpt-5.4-mini")
await anthropic_tracked.generate(prompt=COST_PROMPT, model="claude-haiku-4-5")
await anthropic_tracked.generate(prompt=COST_PROMPT, model="claude-sonnet-4-6")
await gemini_tracked.generate(prompt=COST_PROMPT, model="gemini-3.1-flash-lite")
await gemini_tracked.generate(prompt=COST_PROMPT, model="gemini-3.1-pro")

print(f"Total cost for {len(tracker.entries())} calls: ${tracker.total_cost():.6f}")
print("Entries persisted to: costs.jsonl")

rows = [
    {
        "provider": e.provider,
        "model": e.model,
        "input_tokens": e.input_tokens,
        "output_tokens": e.output_tokens,
        "cost_usd": f"${e.cost_usd:.6f}",
        "latency_ms": e.latency_ms,
    }
    for e in tracker.entries()
]

pd.DataFrame(rows)

Total cost for 6 calls: $0.001742
Entries persisted to: costs.jsonl


,provider,model,input_tokens,output_tokens,cost_usd,latency_ms
0,openai,gpt-5.4-nano,17,40,$0.000053,1271
1,openai,gpt-5.4-mini,17,39,$0.000188,1596
2,anthropic,claude-haiku-4-5,20,43,$0.000235,1699
3,anthropic,claude-sonnet-4-6,20,46,$0.000750,1930
4,gemini,gemini-3.1-flash-lite-preview,11,31,$0.000049,7154
5,gemini,gemini-3.1-pro-preview,11,37,$0.000466,5941
